# Analysis


Ensure that Docker is running!


## Parameters


In [ ]:
MEMORY_BACKEND = "mem0"        # "mem0", "graphiti", "rag", "full_context"
NETWORK_PROFILE = "unconstrained"  # "unconstrained" or "constrained"

CONVERSATION_INDEX = 0          # 0, 1, 2 (LoCoMo conversations)
NO_OF_SESSIONS = 19             # 1-19

# Edge SLM used by the coordinator for tool-calling orchestration.
# Switch between runs to compare models; model name is embedded in output CSV filenames.
#   qwen3:8b      → Alibaba Qwen3 8B  (native tool-calling, thinking budget)
#   ministral:8b  → Mistral Ministral 8B (JSON-strict function calling)
OLLAMA_MODEL = "qwen3:8b"

# --- Run control ---
# RESET_MEMORY_BACKENDS=True  → wipe DB volumes (fresh experiment, new backend)
# RESET_MEMORY_BACKENDS=False → keep existing data (re-run Q&A on already-loaded memories)
RESET_MEMORY_BACKENDS = True

# REBUILD=True  → rebuild Docker images (first run, or after code changes)
# REBUILD=False → skip build, just restart containers with updated env vars
REBUILD = True

# ⚠ DEBUG ONLY — set True to save retrieved memories in the CSV.
# Useful to diagnose IDK responses; leave False for production runs (large CSVs).
LOG_RETRIEVED_MEMORY = True

# Sliding window for fair ingestion (mirrors paper methodology):
#   mem0      → 10  (Mem0 paper: m=10 preceding messages as context)
#   graphiti  → 4   (Zep paper: n=4 messages for entity extraction)
#   rag       → 0   (chunk-based, no windowing needed)
#   full_context → 0
MEMORY_WINDOW_SIZES = {"mem0": 10, "graphiti": 4, "rag": 0, "full_context": 0}
MEMORY_WINDOW_SIZE = MEMORY_WINDOW_SIZES.get(MEMORY_BACKEND, 0)


In [ ]:
NETWORK_PROFILE_UNCONSTRAINED = {
    "TOXIC_LATENCY": 0,
    "TOXIC_JITTER": 0,
    "TOXIC_BANDWIDTH": 0,
    "TOXIC_SLOW_CLOSE": 0,
    "TOXIC_TIMEOUT": 0,
    "TOXIC_SLICER": 0,
    "TOXIC_LIMIT_DATA": 0,
    "TOXIC_RESET_PEER": 0,
}

NETWORK_PROFILE_CONSTRAINED = {
    "TOXIC_LATENCY": 200, # ms
    "TOXIC_JITTER": 50, # ms
    "TOXIC_BANDWIDTH": 1000, # KB/s ≈ 1MB/s ≈ 8Mbps
    "TOXIC_SLOW_CLOSE": 0,
    "TOXIC_TIMEOUT": 0,
    "TOXIC_SLICER": 0,
    "TOXIC_LIMIT_DATA": 0,
    "TOXIC_RESET_PEER": 0,
}

if NETWORK_PROFILE == "constrained":
    PROFILE = NETWORK_PROFILE_CONSTRAINED
else:
    PROFILE = NETWORK_PROFILE_UNCONSTRAINED
PROFILE

## Dependencies


In [ ]:
%pip install python-dotenv


In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file in parent directory
env_path = os.path.join(os.path.dirname(os.getcwd()), '.env')
load_dotenv(env_path)

# Verify critical variables are loaded
print(f"TOXIPROXY_URL: {os.getenv('TOXIPROXY_URL', 'NOT SET')}")
print(f"LOCOMO_URL: {os.getenv('LOCOMO_URL', 'NOT SET')}")
print(f"MEMORY_BACKEND: {os.getenv('MEMORY_BACKEND', 'NOT SET')}")


In [ ]:
%pip install sentence_transformers

## Reset


In [ ]:
import subprocess

if RESET_MEMORY_BACKENDS:
    print("Full reset: stopping containers and wiping DB volumes...")
    subprocess.run(["make", "-C", "../", "reset"], check=True)
else:
    print("Soft reset: stopping containers, keeping DB volumes...")
    subprocess.run(["make", "-C", "../", "down"], check=True)


## Build


In [ ]:
from utils import set_env_var

set_env_var('MEMORY_BACKEND', MEMORY_BACKEND)
set_env_var('MEMORY_WINDOW_SIZE', str(MEMORY_WINDOW_SIZE))
set_env_var('OLLAMA_MODEL', OLLAMA_MODEL)
print(f'MEMORY_BACKEND={MEMORY_BACKEND}, MEMORY_WINDOW_SIZE={MEMORY_WINDOW_SIZE}, OLLAMA_MODEL={OLLAMA_MODEL}')


In [ ]:
import subprocess

if REBUILD:
    print("Building images (set REBUILD=False to skip on subsequent runs)...")
    subprocess.run(["make", "-C", "../", "build-monitoring"], check=True)
    subprocess.run(["make", "-C", "../", "build-proxy"], check=True)
    subprocess.run(["make", "-C", "../", "build-dmas"], check=True)
else:
    print("Skipping build (REBUILD=False) — env vars will be picked up on make up.")


## Run


In [ ]:
!make -C ../ up

In [ ]:
from utils import verify_memory_backend

verify_memory_backend()

## Toxiproxy


In [ ]:
import os

TOXIPROXY_URL = os.environ.get("TOXIPROXY_URL")
TOXIPROXY_URL

In [ ]:
from utils import apply_network_profile

apply_network_profile(PROFILE)

In [ ]:
from utils import check_toxics

check_toxics(PROFILE, TOXIPROXY_URL)

## Memory


(Optional) Run `docker logs memory -f` in a terminal


### Load


In [ ]:
from utils import load_memories

memories = load_memories(NO_OF_SESSIONS, CONVERSATION_INDEX, MEMORY_BACKEND)

### Warmup (Graphiti only)

> After loading all sessions, Graphiti/Neo4j runs async background processing to build the knowledge graph. Starting Q&A before it finishes causes false IDK responses. The warmup time is a paper result demonstrating graph DB impracticality for real-time DMAS.


In [ ]:
from utils import wait_for_graphiti_warmup

# Blocks until Neo4j cloud CPU drops to idle (only for graphiti backend).
# warmup_s is a paper metric: how long the system is unavailable after ingestion.
warmup_s = wait_for_graphiti_warmup(MEMORY_BACKEND)
print(f"Warmup elapsed: {warmup_s:.0f}s")


In [ ]:
import requests
import os

LOCOMO_URL = os.getenv("LOCOMO_URL")
questions = requests.get(f"{LOCOMO_URL}/conversations/index/{CONVERSATION_INDEX}/questions").json()
print(questions)

### Ask

In [ ]:
from utils import run_qa

qa = run_qa(questions, MEMORY_BACKEND, CONVERSATION_INDEX, PROFILE,
           log_retrieved_memory=LOG_RETRIEVED_MEMORY)
